# Agent 03: Purchases Costing - Data Preparation
This notebook is responsible for taking a raw purchases list, standardizing it, and matching it against the canonical SKUs in the database.

**Workflow:**
1.  **Phase 1: Setup and Load Purchases Data** - Load the raw CSV file.
2.  **Phase 2: Fetch Canonical SKUs from Database** - Connect to the production database and retrieve the master SKU list.
3.  **Phase 3: AI-Powered Quantity Standardization** - Clean and standardize the messy `CANTIDAD` column.
4.  **Phase 4: AI-Powered SKU Matching** - Match purchased items to the canonical SKU list.
5.  **Phase 5: Enrich with Supplier and Price Data** - Add supplier information and last purchase prices.
6.  **Phase 6: Export for Human Review** - Create an Excel file for manual validation of all AI proposals.
7.  **Phase 7: Finalize Clean Data** - Load the validated data and produce the final, clean output file.



In [1]:
# Phase 1: Setup and Load Purchases Data
import os
import sys
import pandas as pd

# --- Setup Project Environment ---
# Add the project's root directory to the Python path
# This allows us to import modules from the 'src' folder
root_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
src_path = os.path.join(root_path, 'AGENTS', 'src')
if src_path not in sys.path:
    sys.path.append(src_path)

# Enable autoreload for easier development
# Any changes in the .py files will be automatically reloaded
%load_ext autoreload
%autoreload 2

# --- IMPORTANT: Autoreload Configuration ---
# This prevents re-running expensive API calls when you modify and re-run cells
# The variables below will persist across cell re-runs, avoiding duplicate API charges

# Global variables to store results and avoid re-running API calls
_cached_quantities_processed = False
_cached_skus_processed = False
_cached_quantities_df = None
_cached_skus_df = None

print("🔄 Autoreload enabled - you can modify code without re-running API calls")
print("💡 Tip: Variables with '_cached_' prefix will persist across cell re-runs")

# --- Configuration ---
# Define the paths for our input and output data
INPUT_DATA_PATH = os.path.join(root_path, 'AGENTS', 'data', 'input')
OUTPUT_DATA_PATH = os.path.join(root_path, 'AGENTS', 'data', 'output')
PURCHASES_FILE = '01_CUATRO PATAS_2026_FEB - COMPRAS.csv'

# --- Load Data ---
# Load the raw purchases data from the CSV file
purchases_df = pd.read_csv(os.path.join(INPUT_DATA_PATH, PURCHASES_FILE))

# --- Verification ---
# Display the first few rows of the DataFrame to confirm it loaded correctly
print(f"Successfully loaded {len(purchases_df)} records from '{PURCHASES_FILE}'.")
purchases_df.head()



🔄 Autoreload enabled - you can modify code without re-running API calls
💡 Tip: Variables with '_cached_' prefix will persist across cell re-runs
Successfully loaded 61 records from '01_CUATRO PATAS_2026_FEB - COMPRAS.csv'.


,CATEGORIA,PRODUCTO,CANTIDAD
0,PROTEINAS,OSTIONES VIVOS,280PZ
1,PROTEINAS,TOCINO AHUMADO,150G
2,PROTEINAS,LOMOS DE JUREL,1PZ (8-9KG)
3,PROTEINAS,HOMBRO DE CERDO,2PZ (8-10KG)
4,ABARROTES,MISO ROJO (AKA MISO),300G


### Phase 2: Fetch Canonical SKUs from Database
Now, we will connect to the main project database and retrieve the list of approved, canonical SKUs. This list will serve as our source of truth for the matching process in a later phase.


In [2]:
# Import the database utility function
from database_connector import get_approved_skus_df

# --- Fetch Data ---
# Retrieve the canonical SKUs from the database
try:
    canonical_skus_df = get_approved_skus_df()
    
    if not canonical_skus_df.empty:
        # Use the 'normalized_description' column as specified
        if 'normalized_description' in canonical_skus_df.columns:
            canonical_sku_list = canonical_skus_df['normalized_description'].unique().tolist()
            
            # --- Verification ---
            print(f"Successfully fetched {len(canonical_sku_list)} unique canonical SKUs from the database.")
            print("Sample SKUs:")
            for sku in canonical_sku_list[:5]:
                print(f"- {sku}")
        else:
            print("ERROR: 'normalized_description' column not found in the database.")
            print(f"Available columns: {canonical_skus_df.columns.tolist()}")
            canonical_sku_list = []
    else:
        print("WARNING: No approved SKUs found in the database.")
        canonical_sku_list = []

except Exception as e:
    print(f"ERROR: Could not fetch SKUs from the database.")
    print(f"Please ensure the database is accessible and 'database_connector.py' is correctly configured.")
    print(f"Details: {e}")
    canonical_sku_list = [] # Ensure the list exists to prevent errors in later cells



Connecting to the database to fetch approved SKus...
Connecting to database: db-postgresql-ams3-84868-do-user-23729043-0.l.db.ondigitalocean.com:25060/defaultdb?sslmode=require
✅ Successfully fetched and cleaned 519 approved SKUs.
Successfully fetched 503 unique canonical SKUs from the database.
Sample SKUs:
- CLORO 20LTS
- TORONJA
- LIMON #300
- SHALLOT
- Servicios Integrales de Logística


### Phase 3: AI-Powered Quantity Standardization
This is the first AI-powered step. The goal is to parse the `CANTIDAD` column, which contains unstructured text like `1PZ (8-9KG)` or `150G`.

The agent will call the Gemini API with a specific prompt to extract two clean pieces of information for each item:
1.  `quantity_standardized`: A single numerical value.
2.  `unit_standardized`: A standard unit of measure (e.g., `KG`, `L`, `PZ`).


In [3]:
# Import the necessary AI and prompt engineering modules
from gemini_connector import call_gemini_api
from prompt_engineering import create_unit_conversion_prompt
from tqdm.auto import tqdm
import json

# Configure tqdm for pandas
tqdm.pandas()

# --- Check if we already processed quantities (avoid duplicate API calls) ---
if _cached_quantities_processed and _cached_quantities_df is not None:
    print("✅ Using cached quantity standardization results (no API call needed)")
    standardized_quantities_df = _cached_quantities_df
else:
    # --- Prepare Data for AI ---
    # Get the unique list of quantities to avoid redundant API calls
    quantities_to_process = purchases_df['CANTIDAD'].unique().tolist()

    print(f"Processing {len(quantities_to_process)} unique quantities for standardization...")

    # --- Generate Prompt ---
    # Create the prompt using our list of quantities
    quantity_prompt = create_unit_conversion_prompt(quantities_to_process)

    print("Sending quantity data to AI for standardization...")
    # --- Call AI Service ---
    # Send the prompt to the Gemini API and get back a structured response
    try:
        ai_response = call_gemini_api(quantity_prompt)
        
        # Parse the JSON response (handle markdown code blocks)
        try:
            # Remove markdown code blocks if present
            if ai_response.strip().startswith('```json'):
                # Extract JSON from markdown code block
                json_start = ai_response.find('{')
                json_end = ai_response.rfind('}') + 1
                if json_start != -1 and json_end > json_start:
                    ai_response = ai_response[json_start:json_end]
            
            standardized_data = json.loads(ai_response)
            
            # Convert the response to a DataFrame for merging
            standardized_rows = []
            for original_quantity, conversion_data in standardized_data.items():
                standardized_rows.append({
                    'original_quantity': original_quantity,
                    'quantity_standardized': conversion_data.get('conversion_factor', 0),
                    'unit_standardized': conversion_data.get('proposed_standard', 'No_aplica'),
                    'conversion_notes': conversion_data.get('notes', '')
                })
            
            standardized_quantities_df = pd.DataFrame(standardized_rows)
            
            # Cache the results to avoid re-running API calls
            _cached_quantities_df = standardized_quantities_df
            _cached_quantities_processed = True
            print("💾 Cached quantity standardization results")

        except json.JSONDecodeError as e:
            print(f"ERROR: Failed to parse AI response as JSON: {e}")
            print(f"Raw AI response: {ai_response}")
            standardized_quantities_df = None

    except Exception as e:
        print(f"ERROR: Failed during AI quantity standardization.")
        print(f"Please check the Gemini API connection and the prompt engineering functions.")
        print(f"Details: {e}")
        standardized_quantities_df = None

# --- Integrate Results (if we have data) ---
if standardized_quantities_df is not None:
    # Merge the standardized quantities back into the main DataFrame
    purchases_df = pd.merge(
        purchases_df,
        standardized_quantities_df,
        left_on='CANTIDAD',
        right_on='original_quantity',
        how='left'
    )
    
    # --- Verification ---
    print("Successfully received and merged standardized quantities.")
    # Display the new columns to verify the results
    display(purchases_df[['CANTIDAD', 'quantity_standardized', 'unit_standardized']].head())
else:
    print("⚠️ No quantity standardization data available")
    purchases_df['quantity_standardized'] = None
    purchases_df['unit_standardized'] = None




Processing 37 unique quantities for standardization...
Sending quantity data to AI for standardization...
✅ Gemini model 'gemini-2.0-flash' initialized successfully.
Submitting prompt to Gemini API...
✅ Received response from API.
💾 Cached quantity standardization results
Successfully received and merged standardized quantities.


,CANTIDAD,quantity_standardized,unit_standardized
0,280PZ,1.000,Piezas
1,150G,0.001,Kilogramos
2,1PZ (8-9KG),1.000,Piezas
3,2PZ (8-10KG),1.000,Piezas
4,300G,0.001,Kilogramos


### Phase 4: AI-Powered SKU Matching
This is the most critical phase. The agent will take the informal `PRODUCTO` names from the purchases list and intelligently match them against the `canonical_sku_list` we retrieved from the database.

This uses an AI-powered "fuzzy matching" process to find the most likely canonical SKU for each purchased item.


In [4]:
# Import the SKU matching prompt function
from prompt_engineering import create_sku_matching_prompt

# --- Check if we already processed SKU matching (avoid duplicate API calls) ---
if _cached_skus_processed and _cached_skus_df is not None:
    print("✅ Using cached SKU matching results (no API call needed)")
    matched_skus_df = _cached_skus_df
else:
    # --- Prepare Data for AI ---
    # Get the unique list of product descriptions to process
    products_to_match = purchases_df['PRODUCTO'].unique().tolist()

    print(f"Processing {len(products_to_match)} unique products for SKU matching...")

    # --- Check if there are SKUs to match against ---
    if canonical_sku_list:
        # Convert SKU list to string for the prompt
        sku_list_as_string = "\n".join([f"- {sku}" for sku in canonical_sku_list])
        
        print("Sending product data to AI for SKU matching...")
        # --- Call AI Service ---
        try:
            # Process each product individually for better matching accuracy
            matched_skus = []
            
            for product in tqdm(products_to_match, desc="Matching products"):
                # Create prompt for this specific product
                sku_matching_prompt = create_sku_matching_prompt(product, sku_list_as_string)
                
                # Get AI response
                ai_response = call_gemini_api(sku_matching_prompt)
                
                try:
                    # Parse JSON response (handle markdown code blocks)
                    # Remove markdown code blocks if present
                    if ai_response.strip().startswith('```json'):
                        # Extract JSON from markdown code block
                        json_start = ai_response.find('{')
                        json_end = ai_response.rfind('}') + 1
                        if json_start != -1 and json_end > json_start:
                            ai_response = ai_response[json_start:json_end]
                    
                    match_result = json.loads(ai_response)
                    best_match = match_result.get('best_match_sku', 'NO_MATCH_FOUND')
                    
                    matched_skus.append({
                        'original_product': product,
                        'canonical_sku_match': best_match
                    })
                    
                except json.JSONDecodeError as e:
                    print(f"ERROR: Failed to parse AI response for '{product}': {e}")
                    matched_skus.append({
                        'original_product': product,
                        'canonical_sku_match': 'PARSING_FAILED'
                    })
            
            # Convert to DataFrame
            matched_skus_df = pd.DataFrame(matched_skus)
            
            # Cache the results to avoid re-running API calls
            _cached_skus_df = matched_skus_df
            _cached_skus_processed = True
            print("💾 Cached SKU matching results")

        except Exception as e:
            print(f"ERROR: Failed during AI SKU matching.")
            print(f"Details: {e}")
            matched_skus_df = None

    else:
        print("WARNING: Canonical SKU list is empty. Skipping SKU matching.")
        print("Please check the database connection in Phase 2.")
        matched_skus_df = None

# --- Integrate Results (if we have data) ---
if matched_skus_df is not None:
    # Merge the matched SKUs back into the main DataFrame
    purchases_df = pd.merge(
        purchases_df,
        matched_skus_df,
        left_on='PRODUCTO',
        right_on='original_product',
        how='left'
    )

    # --- Verification ---
    print("Successfully received and merged SKU matches.")
    # Display the new column to verify the results
    display(purchases_df[['PRODUCTO', 'canonical_sku_match']].head())
else:
    print("⚠️ No SKU matching data available")
    purchases_df['canonical_sku_match'] = 'SKIPPED_NO_SKUS'



Processing 61 unique products for SKU matching...
Sending product data to AI for SKU matching...


Matching products:   0%|          | 0/61 [00:00<?, ?it/s]

Submitting prompt to Gemini API...
✅ Received response from API.
Submitting prompt to Gemini API...
✅ Received response from API.
Submitting prompt to Gemini API...
✅ Received response from API.
Submitting prompt to Gemini API...
✅ Received response from API.
Submitting prompt to Gemini API...
✅ Received response from API.
Submitting prompt to Gemini API...
✅ Received response from API.
Submitting prompt to Gemini API...
✅ Received response from API.
Submitting prompt to Gemini API...
✅ Received response from API.
Submitting prompt to Gemini API...
✅ Received response from API.
Submitting prompt to Gemini API...
✅ Received response from API.
Submitting prompt to Gemini API...
✅ Received response from API.
Submitting prompt to Gemini API...
✅ Received response from API.
Submitting prompt to Gemini API...
✅ Received response from API.
Submitting prompt to Gemini API...
✅ Received response from API.
Submitting prompt to Gemini API...
✅ Received response from API.
Submitting prompt to Gemi

,PRODUCTO,canonical_sku_match
0,OSTIONES VIVOS,NO_MATCH_FOUND
1,TOCINO AHUMADO,TOCINO
2,LOMOS DE JUREL,NO_MATCH_FOUND
3,HOMBRO DE CERDO,JAMON CERDO KOWI
4,MISO ROJO (AKA MISO),NO_MATCH_FOUND


### Phase 5: Enrich with Supplier and Price Data
Now that we have matched products to canonical SKUs, let's enrich the data with supplier information and last purchase prices from the database.

This phase will:
1. Look up supplier details for each matched SKU
2. Retrieve the last purchase price for each SKU
3. Add this information to our final dataset


In [10]:
# Phase 5: Enrich with Supplier and Price Data
print("🔍 Enriching data with supplier and price information...")

# First, let's check what columns are available in our canonical SKUs data
if 'canonical_sku_list' in globals() and canonical_sku_list:
    print(f"📊 We have {len(canonical_sku_list)} canonical SKUs to work with")
    
    # Get the matched SKUs from our purchases data
    matched_skus = purchases_df['canonical_sku_match'].dropna().unique().tolist()
    matched_skus = [sku for sku in matched_skus if sku not in ['NO_MATCH_FOUND', 'PARSING_FAILED', 'MATCHING_FAILED', 'SKIPPED_NO_SKUS']]
    
    print(f"📋 Found {len(matched_skus)} successfully matched SKUs")
    
    if matched_skus:
        # Create a lookup DataFrame from the canonical SKUs
        # We need to get the full canonical_skus_df with all columns
        try:
            # Re-fetch the full canonical SKUs data with all columns
            full_canonical_df = get_approved_skus_df()
            
            if not full_canonical_df.empty:
                print(f"📋 Canonical SKUs table has {len(full_canonical_df)} records")
                print(f"📋 Available columns: {list(full_canonical_df.columns)}")
                
                # Filter to only the matched SKUs
                matched_skus_df = full_canonical_df[
                    full_canonical_df['normalized_description'].isin(matched_skus)
                ].copy()
                
                print(f"📋 Found {len(matched_skus_df)} records for matched SKUs")
                
                if not matched_skus_df.empty:
                    # Display sample of what we found
                    print("\n📋 Sample matched SKU data:")
                    display(matched_skus_df.head())
                    
                    # Merge supplier and price data back to purchases
                    purchases_df = pd.merge(
                        purchases_df,
                        matched_skus_df[['normalized_description'] + [col for col in matched_skus_df.columns if col != 'normalized_description']],
                        left_on='canonical_sku_match',
                        right_on='normalized_description',
                        how='left',
                        suffixes=('', '_sku')
                    )
                    
                    print("✅ Successfully enriched data with supplier and price information")
                    print(f"📊 Final dataset shape: {purchases_df.shape}")
                    
                    # Show sample of enriched data
                    print("\n📋 Sample enriched data:")
                    display(purchases_df[['PRODUCTO', 'canonical_sku_match', 'quantity_standardized', 'unit_standardized']].head())
                    
                else:
                    print("⚠️ No matching records found in canonical SKUs table")
                    
            else:
                print("❌ Could not retrieve canonical SKUs data")
                
        except Exception as e:
            print(f"❌ Error enriching data: {e}")
    else:
        print("⚠️ No successfully matched SKUs to enrich")
else:
    print("⚠️ No canonical SKU list available for enrichment")


🔍 Enriching data with supplier and price information...
📊 We have 503 canonical SKUs to work with
📋 Found 49 successfully matched SKUs
Connecting to the database to fetch approved SKus...
Connecting to database: db-postgresql-ams3-84868-do-user-23729043-0.l.db.ondigitalocean.com:25060/defaultdb?sslmode=require
❌ ERROR: Failed to fetch approved SKUs: (psycopg2.OperationalError) connection to server at "db-postgresql-ams3-84868-do-user-23729043-0.l.db.ondigitalocean.com" (134.209.84.88), port 25060 failed: server closed the connection unexpectedly
	This probably means the server terminated abnormally
	before or while processing the request.

(Background on this error at: https://sqlalche.me/e/20/e3q8)
❌ Could not retrieve canonical SKUs data


### Phase 6: Export for Human Review
This is the single, most important checkpoint. The agent will now export an Excel file containing the original data alongside all of the AI's proposed standardizations and matches.

**Your task is to open this file, review the AI-generated columns, and make any necessary corrections.** Once you save the file, the final phase of the notebook will load your approved data.


In [8]:
# --- Configuration for Review File ---
REVIEW_FILE = '02_purchases_for_review.xlsx'
review_file_path = os.path.join(OUTPUT_DATA_PATH, REVIEW_FILE)

# --- Select and Order Columns for Clarity ---
# Define the order of columns to make the review process intuitive
columns_for_review = [
    # Original Data
    'CATEGORIA',
    'PRODUCTO',
    'CANTIDAD',
    
    # AI Proposals for Review
    'quantity_standardized',
    'unit_standardized',
    'canonical_sku_match',
    
    # Drop helper columns if they exist
    'original_quantity', 
    'original_product'
]

# Filter the DataFrame to only include the columns we need for review
review_df = purchases_df[[col for col in columns_for_review if col in purchases_df.columns]]

# --- Export to Excel ---
try:
    review_df.to_excel(review_file_path, index=False)
    print(f"✅ Successfully exported data for your review.")
    print(f"Please open the following file, make your corrections, and save it:")
    print(f" -> {review_file_path}")

except Exception as e:
    print(f"ERROR: Could not export the review file.")
    print(f"Details: {e}")




✅ Successfully exported data for your review.
Please open the following file, make your corrections, and save it:
 -> c:\Users\Admin\Desktop\02. PROGRAMACION\01. TRAINING EXCERSISES\01. Proyecto Costeo Facturas_v3\recipe_analyzer\AGENTS\data\output\02_purchases_for_review.xlsx


### Phase 7: Finalize Clean Data
**This is the final step.**

Run this cell only *after* you have reviewed and saved your changes in the `02_purchases_for_review.xlsx` file. This will load your human-approved data and save it as a final, clean file, completing the agent's task.


In [6]:
# --- Configuration for Final File ---
FINAL_FILE = '03_purchases_clean_and_matched.xlsx'
final_file_path = os.path.join(OUTPUT_DATA_PATH, FINAL_FILE)

# --- Load Human-Reviewed Data ---
try:
    print(f"Loading your validated data from '{REVIEW_FILE}'...")
    final_df = pd.read_excel(review_file_path)

    # --- Verification ---
    print("Performing final validation...")
    # Check for any missing values in the key columns after review
    missing_data = final_df[['quantity_standardized', 'unit_standardized', 'canonical_sku_match']].isnull().sum()
    if missing_data.sum() > 0:
        print("\nWARNING: Missing data found in the reviewed file.")
        print(missing_data)
    else:
        print("✅ Final validation passed. No missing data.")

    # --- Export Final Data ---
    final_df.to_excel(final_file_path, index=False)
    print(f"\n🚀 Agent complete! The final, cleaned data has been saved to:")
    print(f" -> {final_file_path}")
    
    display(final_df.head())

except FileNotFoundError:
    print(f"ERROR: The review file was not found.")
    print(f"Please ensure the file exists at the correct path: {review_file_path}")
except Exception as e:
    print(f"ERROR: An unexpected error occurred while finalizing the data.")
    print(f"Details: {e}")



Loading your validated data from '02_purchases_for_review.xlsx'...
Performing final validation...
✅ Final validation passed. No missing data.

🚀 Agent complete! The final, cleaned data has been saved to:
 -> c:\Users\Admin\Desktop\02. PROGRAMACION\01. TRAINING EXCERSISES\01. Proyecto Costeo Facturas_v3\recipe_analyzer\AGENTS\data\output\03_purchases_clean_and_matched.xlsx


,CATEGORIA,PRODUCTO,CANTIDAD,quantity_standardized,unit_standardized,canonical_sku_match,original_quantity,original_product
0,PROTEINAS,OSTIONES VIVOS,280PZ,1.000,Piezas,NO_MATCH_FOUND,280PZ,OSTIONES VIVOS
1,PROTEINAS,TOCINO AHUMADO,150G,0.001,Kilogramos,TOCINO,150G,TOCINO AHUMADO
2,PROTEINAS,LOMOS DE JUREL,1PZ (8-9KG),1.000,Piezas,NO_MATCH_FOUND,1PZ (8-9KG),LOMOS DE JUREL
3,PROTEINAS,HOMBRO DE CERDO,2PZ (8-10KG),1.000,Piezas,JAMON CERDO KOWI,2PZ (8-10KG),HOMBRO DE CERDO
4,ABARROTES,MISO ROJO (AKA MISO),300G,0.001,Kilogramos,NO_MATCH_FOUND,300G,MISO ROJO (AKA MISO)


In [7]:
# DIAGNOSTIC: Let's debug the database connection issue
print("🔍 DIAGNOSTIC: Checking database connection and SKU data...")
print("=" * 60)

# Re-run the database connection with more detailed output
try:
    canonical_skus_df = get_approved_skus_df()
    
    print(f"📊 Database connection result:")
    print(f"   - DataFrame shape: {canonical_skus_df.shape}")
    print(f"   - Is empty: {canonical_skus_df.empty}")
    
    if not canonical_skus_df.empty:
        print(f"   - Columns available: {list(canonical_skus_df.columns)}")
        
        # Check if 'normalized_description' column exists
        if 'normalized_description' in canonical_skus_df.columns:
            print(f"   - ✅ Found 'normalized_description' column!")
            canonical_sku_list = canonical_skus_df['normalized_description'].unique().tolist()
            print(f"   - Unique normalized descriptions: {len(canonical_sku_list)}")
            print(f"   - Sample normalized descriptions:")
            for i, desc in enumerate(canonical_sku_list[:10]):
                print(f"     {i+1}. {desc}")
        else:
            print(f"   - ❌ 'normalized_description' column not found!")
            print(f"   - Available columns: {list(canonical_skus_df.columns)}")
        
        print(f"\n📋 COMPLETE TABLE:")
        print("=" * 80)
        # Display the full table
        pd.set_option('display.max_columns', None)
        pd.set_option('display.max_rows', None)
        pd.set_option('display.width', None)
        pd.set_option('display.max_colwidth', 50)
        print(canonical_skus_df)
        
    else:
        print("   - ❌ DataFrame is empty - no data retrieved from database")
        
except Exception as e:
    print(f"   - ❌ Database connection failed: {e}")
    print(f"   - Error type: {type(e).__name__}")

print("\n" + "=" * 60)


🔍 DIAGNOSTIC: Checking database connection and SKU data...
Connecting to the database to fetch approved SKus...
Connecting to database: db-postgresql-ams3-84868-do-user-23729043-0.l.db.ondigitalocean.com:25060/defaultdb?sslmode=require
✅ Successfully fetched and cleaned 519 approved SKUs.
📊 Database connection result:
   - DataFrame shape: (519, 24)
   - Is empty: False
   - Columns available: ['id', 'sku_key', 'product_code', 'internal_code', 'normalized_description', 'category', 'subcategory', 'sub_sub_category', 'standardized_unit', 'correct_unit_code', 'units_per_package', 'package_type', 'conversion_notes', 'typical_quantity_range', 'approved_by', 'approval_date', 'confidence_score', 'usage_count', 'last_used', 'review_status', 'review_notes', 'created_at', 'updated_at', 'client_rfc']
   - ✅ Found 'normalized_description' column!
   - Unique normalized descriptions: 503
   - Sample normalized descriptions:
     1. CLORO 20LTS
     2. TORONJA
     3. LIMON #300
     4. SHALLOT
    